In [ ]:
import glob
import os
import re
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from sklearn.manifold import TSNE


font_path = './AUPassata_Rg.ttf'
fm.fontManager.addfont(font_path)


def init_plotting():
    plt.rcParams['figure.figsize'] = (8, 6)
    plt.rcParams['font.size'] = 10
    plt.rcParams['font.family'] = 'AU Passata'
    plt.rcParams['axes.labelsize'] = plt.rcParams['font.size']
    plt.rcParams['axes.titlesize'] = 1.5 * plt.rcParams['font.size']
    plt.rcParams['legend.fontsize'] = plt.rcParams['font.size']
    plt.rcParams['xtick.labelsize'] = plt.rcParams['font.size']
    plt.rcParams['ytick.labelsize'] = plt.rcParams['font.size']
    plt.rcParams['savefig.dpi'] = 200
    plt.rcParams['xtick.major.size'] = 3
    plt.rcParams['xtick.major.width'] = 1
    plt.rcParams['ytick.major.size'] = 3
    plt.rcParams['ytick.major.width'] = 1
    plt.rcParams['legend.frameon'] = False
    plt.rcParams['axes.linewidth'] = 1


init_plotting()

with open('../confs/thesis.yaml') as f:
    config = yaml.safe_load(f)

# ACCESSIBILITY_DIR = f"../{config['data']['accessibility_scores_dir']}"
# ACCESSIBILITY_DIR = '../vae_part_of_delfi1_only_pos_lymphoid_72x192/accessibility_scores/'
# METADATA_PATH = f"../{config['data']['inference_metadata_path']}"
# METADATA_PATH = '../vae_part_of_delfi1_training_raw_data/meta/inference.tsv'
# METADATA_PATH = '../vae_full_delfi1_training_raw_data/meta/DELFI_LUCAS_balanced.tsv'

ACCESSIBILITY_DIR = '../vae_extra_72x192_healthy_all/accessibility_scores/'
# METADATA_PATH = '../vae_part_of_delfi1_training_raw_data/meta/inference.tsv'
METADATA_PATH = '../vae_full_delfi1_training_raw_data/meta/DELFI_LUCAS_balanced.tsv'

meta_df = pd.read_csv(METADATA_PATH, sep='\t', dtype=str)
metadata_map = {
    row['Patient']: row['Patient type']
    for _, row in meta_df.iterrows()
}

# load data
SID_PATTERN = r'(.+?)__(.+?)_latent\.npz$'
npz_files = sorted(glob.glob(os.path.join(ACCESSIBILITY_DIR, '*_latent.npz')))

records = []
for path in npz_files:
    m = re.match(SID_PATTERN, os.path.basename(path))
    if not m:
        continue
    sid, dhs = m.group(1), m.group(2)
    if sid not in metadata_map:
        continue
    data = np.load(path)
    records.append({
        'path': path,
        'sample': sid,
        'dhs': dhs,
        'mu': np.asarray(data['mu']),
    })

raw_df = pd.DataFrame(records)


def plot_embedding(ax, df, x, y, group_col, color_map, title):
    for label, group in df.groupby(group_col):
        ax.scatter(
            group[x], group[y],
            label=label, c=[color_map[label]], alpha=0.7, s=30,
            edgecolors='white', linewidths=0.3,
        )
    ax.set_xlabel(x)
    ax.set_ylabel(y)
    ax.set_title(title)


def add_binary_labels(df):
    df = df.copy()
    df['disease'] = df['sample'].map(metadata_map)
    df['binary'] = df['disease'].str.lower().apply(
        lambda x: 'healthy' if x == 'healthy' else 'cancer'
    )
    return df


def run_tsne(mu_matrix, random_state=42, min_perplexity=30):
    perplexity = min(min_perplexity, len(mu_matrix) - 1)
    tsne = TSNE(n_components=2, perplexity=perplexity, random_state=random_state)
    return tsne.fit_transform(mu_matrix)


binary_colors = {'healthy': 'red', 'cancer': 'blue'}

In [ ]:
# all DHS concatenated, one point per patient
df_all = raw_df.sort_values(['sample', 'dhs']).reset_index(drop=True)

dhs_counts = df_all.groupby('sample')['dhs'].nunique()
expected_dhs = dhs_counts.max()
incomplete = dhs_counts[dhs_counts != expected_dhs]
if len(incomplete) > 0:
    print(f"Warning: dropping {len(incomplete)} samples with incomplete DHS coverage")
    df_all = df_all[~df_all['sample'].isin(incomplete.index)]

patient_mu = df_all.groupby('sample')['mu'].apply(
    lambda s: np.concatenate(s.values)
)
mu_matrix_all = np.vstack(patient_mu.values)

df_concat = pd.DataFrame({'sample': patient_mu.index.tolist()})
df_concat = add_binary_labels(df_concat)
coords = run_tsne(mu_matrix_all, min_perplexity=30)
df_concat['tsne_1'], df_concat['tsne_2'] = coords[:, 0], coords[:, 1]

fig, ax = plt.subplots(figsize=(8, 6))
plot_embedding(ax, df_concat, 'tsne_1', 'tsne_2', 'binary', binary_colors,
               't-SNE: Healthy vs Cancerous (all DHS concatenated)')
ax.legend()
plt.tight_layout()
# plt.savefig('figures/result_04_within_cohort_tsne.png', dpi=300)
plt.show()

In [ ]:
# single DHS only
def plot_single_dhs(dhs_name):
    sub = raw_df[raw_df['dhs'] == dhs_name].copy()
    if len(sub) == 0:
        available = sorted(raw_df['dhs'].unique())
        raise ValueError(f"No samples found for DHS '{dhs_name}'. Available: {available}")

    sub = sub.sort_values('sample').reset_index(drop=True)
    mu_matrix = np.vstack(sub['mu'].values)

    df_single = sub[['sample']].copy()
    df_single = add_binary_labels(df_single)
    coords = run_tsne(mu_matrix)
    df_single['tsne_1'], df_single['tsne_2'] = coords[:, 0], coords[:, 1]

    _, ax = plt.subplots(figsize=(8, 6))
    plot_embedding(ax, df_single, 'tsne_1', 'tsne_2', 'binary', binary_colors,
                   f't-SNE: Healthy vs Cancerous ({dhs_name} only)')
    ax.legend()
    plt.tight_layout()
    plt.show()
    return df_single


df_lymphoid = plot_single_dhs('Lymphoid')

In [ ]:
import glob
import os
import re
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml


font_path = './AUPassata_Rg.ttf'
fm.fontManager.addfont(font_path)


def init_plotting():
  plt.rcParams['figure.figsize'] = (8, 6)
  plt.rcParams['font.size'] = 10
  plt.rcParams['font.family'] = 'AU Passata'
  plt.rcParams['axes.labelsize'] = plt.rcParams['font.size']
  plt.rcParams['axes.titlesize'] = 1.5 * plt.rcParams['font.size']
  plt.rcParams['legend.fontsize'] = plt.rcParams['font.size']
  plt.rcParams['xtick.labelsize'] = plt.rcParams['font.size']
  plt.rcParams['ytick.labelsize'] = plt.rcParams['font.size']
  plt.rcParams['savefig.dpi'] = 200
  plt.rcParams['xtick.major.size'] = 3
  plt.rcParams['xtick.major.width'] = 1
  plt.rcParams['ytick.major.size'] = 3
  plt.rcParams['ytick.major.width'] = 1
  plt.rcParams['legend.frameon'] = False
  plt.rcParams['axes.linewidth'] = 1


init_plotting()

with open('../confs/thesis.yaml') as f:
  config = yaml.safe_load(f)

# ACCESSIBILITY_DIR = f"../{config['data']['accessibility_scores_dir']}"
# METADATA_PATH = f"../{config['data']['inference_metadata_path']}"
ACCESSIBILITY_DIR = '../vae_within_cohort_72x192_healthy_cancer_lymphoid/accessibility_scores/'
# METADATA_PATH = '../vae_full_delfi1_training_raw_data/meta/DELFI_LUCAS_balanced.tsv'
METADATA_PATH = '../vae_part_of_delfi1_training_raw_data/meta/inference.tsv'

meta_df = pd.read_csv(METADATA_PATH, sep='\t', dtype=str)
metadata_map = {
  row['Patient']: row['Patient type']
  for _, row in meta_df.iterrows()
}

SID_PATTERN = r'(.+?)__(.+?)_latent\.npz$'
npz_files = sorted(glob.glob(os.path.join(ACCESSIBILITY_DIR, '*Lymphoid_latent.npz')))

records = []
for path in npz_files:
  m = re.match(SID_PATTERN, os.path.basename(path))
  if not m:
      continue
  sid, dhs = m.group(1), m.group(2)
  if sid not in metadata_map:
      continue
  data = np.load(path)
  mu = np.asarray(data['mu']).ravel()
  records.append({
      'sample': sid,
      'dhs': dhs,
      'mu_1': float(mu[0]),
      'mu_2': float(mu[1]),
  })

df = pd.DataFrame(records)
df['disease'] = df['sample'].map(metadata_map)
df['binary'] = df['disease'].str.lower().apply(
  lambda x: 'healthy' if x == 'healthy' else 'cancer'
)

binary_colors = {'healthy': 'red', 'cancer': 'blue'}

fig, ax = plt.subplots(figsize=(8, 6))
for label, group in df.groupby('binary'):
  ax.scatter(
      group['mu_1'], group['mu_2'],
      label=label, c=[binary_colors[label]], alpha=0.7, s=30,
      edgecolors='white', linewidths=0.3,
  )
ax.set_xlabel('mu_1')
ax.set_ylabel('mu_2')
ax.set_title('VAE 2D latent space: Healthy vs Cancerous (Lymphoid DHS)')
ax.legend()
plt.tight_layout()
plt.savefig('figures/result_04_within_cohort_mu_values.png', dpi=300)
plt.show()
